In [1]:
import json
import os
import typing as t
from datetime import datetime

import numpy as np
import pandas as pd
from s3fs import S3FileSystem


from toast_cap import LOGGER
from toast_cap.utilities import config
from toast_cap.utilities.dao import DataWarehouseConnection
from toast_cap.utilities.functions import split_ids_by_range, timestamp_str

import os
os.chdir('../')

In [2]:
## change values in this cell depending on your requirements
run_date_str = '20250131'  # queries will pull data up to 1 day prior to run_date_str
volume_start_dt = '20200401' # pull daily revenue data starting from this date
output_dir = f's3://toast-datascience-sandbox/PradeepA/Early_Defaults/{run_date_str}' # directory to store the raw data
transaction_output_dir = os.path.join(output_dir, 'daily_data')  # directory to store the daily revenue data parquet files
os.environ["SNOWFLAKE_PRIVATE_KEY_PASSWORD"] = ""

In [3]:
partial_days = 1
num_days_volume_data = (pd.to_datetime(run_date_str) - pd.to_datetime(volume_start_dt)).days + 1 - partial_days

datetime_now = datetime.strptime(run_date_str, "%Y%m%d")
end_date = timestamp_str("%Y%m%d", days=partial_days, time_now=datetime_now)

ts_start_date = timestamp_str(
    "%Y%m%d",
    days=partial_days + num_days_volume_data - 1,
    time_now=datetime_now
)

In [269]:
## acct data
sql_acct = """
        WITH first_loan as (
        SELECT TOASTORDERS_RESTAURANT_ID,
        MIN(created_date) AS first_loan_date,
        MIN(case when term_in_days=90 then created_date else NULL end) AS first_loan_date_90d,
        MIN(case when term_in_days=270 then created_date else NULL end) AS first_loan_date_270d,
        MIN(case when term_in_days=360 then created_date else NULL end) AS first_loan_date_360d
        FROM toast_capital.product_record
        WHERE product_type in ('FIXED_FEE_LOAN', 'FIXED_FEE_WINTER_LOAN', 'MCA')
        AND funding_partner IN ('Toast', 'WebBank')
        AND created_date <= TO_DATE(TO_VARCHAR({}), 'YYYYMMDD')
        GROUP BY 1
        )
        
        SELECT CAST(cust.ACCOUNT_TOAST_ID AS int) AS "rid"
            ,cust.ACCOUNT_TOAST_GUID AS "restaurant_guid"
            ,cust.TOASTORDERS_STATE AS "state"
            ,cust.CUSTOMER_STATUS AS "customer_status"
            ,cust.FIRST_ORDER_DATE AS "first_order_date"
            ,cust.CHURN_DATE AS "churn_date"
            ,cust.CHURN_REASON AS "churn_reason"
            ,cust.account_restaurant_category as "account_restaurant_category"
            ,cust.restaurant_type as "restaurant_type"
            ,cust.parent_market_segment as "parent_market_segment"
            ,tx.MIN_DATE as "tx_date_min"
            ,tx.MAX_DATE as "tx_date_max"
            ,tx.GPV_MIN_DATE as "gpv_date_min"
            ,tx.GPV_MAX_DATE as "gpv_date_max"
            ,l.first_loan_date as "first_loan_date"
            ,l.first_loan_date_90d as "first_loan_date_90d"
            ,l.first_loan_date_270d as "first_loan_date_270d"
            ,l.first_loan_date_360d as "first_loan_date_360d"
        FROM ANALYTICS_CORE.CUSTOMER cust
        LEFT JOIN (
            SELECT op.TOASTORDERS_RESTAURANT_ID
                ,to_date(to_char(min(op.date_id)), 'YYYYMMDD') AS MIN_DATE
                ,to_date(to_char(max(op.date_id)), 'YYYYMMDD') AS MAX_DATE
                ,to_date(to_char(min(CASE WHEN op.PAYMENT_TYPE = 'CREDIT'
                                     THEN op.date_id ELSE NULL END)), 'YYYYMMDD') AS GPV_MIN_DATE
                ,to_date(to_char(max(CASE WHEN op.PAYMENT_TYPE = 'CREDIT'
                                     THEN op.date_id ELSE NULL END)), 'YYYYMMDD') AS GPV_MAX_DATE
            FROM PRODUCT_TRANSACTIONS.TOAST_TRANSACTION_VOLUME_HOURLY op
            WHERE op.TOASTORDERS_RESTAURANT_ID IS NOT NULL
            AND op.date_id between {} and {}
            GROUP BY op.TOASTORDERS_RESTAURANT_ID
            ) AS tx ON cust.ACCOUNT_TOAST_ID = tx.TOASTORDERS_RESTAURANT_ID
        LEFT JOIN first_loan l on cust.ACCOUNT_TOAST_ID = l.TOASTORDERS_RESTAURANT_ID 
        WHERE cust.FIRST_ORDER_DATE IS NOT NULL
            AND cust.ACCOUNT_TOAST_ID IS NOT NULL
"""

# accts = DataWarehouseConnection().query(sql_acct.format(end_date, ts_start_date, end_date))

accts = DataWarehouseConnection().query(sql_acct.format(end_date, ts_start_date, end_date))
for x in ['first_loan_date', 'first_loan_date_90d', 'first_loan_date_270d', 'first_loan_date_360d',
         'first_order_date', 'churn_date', 'tx_date_min', 'tx_date_max', 'gpv_date_min', 'gpv_date_max']:
    accts[x] = pd.to_datetime(accts[x])
accts.to_parquet(os.path.join(output_dir, 'accts.parquet'))
accts.head(1)

,rid,restaurant_guid,state,customer_status,first_order_date,churn_date,churn_reason,account_restaurant_category,restaurant_type,parent_market_segment,tx_date_min,tx_date_max,gpv_date_min,gpv_date_max,first_loan_date,first_loan_date_90d,first_loan_date_270d,first_loan_date_360d
0,162379000000000000,01d1c001-6c46-4061-b72a-cb5a4799bf55,FL,Churned,2023-08-08,2025-01-27,Permanently Closed,QSR - Fast Casual,Pizzeria,SMB,2023-08-08,2024-11-24,2023-08-08,2024-11-24,NaT,NaT,NaT,NaT


In [270]:
df_rid_guid_bridge = accts[['rid', 'restaurant_guid']].drop_duplicates(keep='last')
df_rid_guid_bridge.head()

,rid,restaurant_guid
0,162379000000000000,01d1c001-6c46-4061-b72a-cb5a4799bf55
1,47445000000000000,4403c864-4c4f-4dfc-897f-31e865819768
2,28468000000000000,6ffde169-53a7-43a2-a94b-e6d40f9f98e4
3,168613000000000000,e19736eb-48f5-4368-a0b2-366dd98f10af
4,197906000000000000,6c92cffa-40bf-4404-930e-b613d19f9d8e


In [271]:
# Group by 'rid' and count unique 'restaurant_guid' values
rid_counts = df_rid_guid_bridge.groupby('rid')['restaurant_guid'].nunique()
guid_counts = df_rid_guid_bridge.groupby('restaurant_guid')['rid'].nunique()
# Count how many 'rid' values have more than one unique 'restaurant_guid'
one_to_many_rid_count = (rid_counts > 1).sum()
one_to_many_guid_count = (rid_counts > 1).sum()

one_to_many_rid_count.shape, one_to_many_guid_count.shape

((), ())

In [272]:
df_rid_guid_bridge.to_parquet(os.path.join(output_dir, 'rx_guid_rid_bridge.parquet'))

In [7]:
## module data
sql_module = '''
            SELECT CAST(r.ACCOUNT_TOAST_ID AS int) AS "restaurant_id",
                CAST(TO_VARCHAR(d.FIRSTDAYOFMONTH, 'YYYYMM') AS int) AS "yearmon",
                m.LIVE_ONLINE_ORDERING_MODULE_COUNT AS "live_online_ordering_module_count",
                m.live_saas_mrr AS "live_saas_mrr",
                m.live_saas_module_count AS "live_saas_module_count",
                m.live_gift_card_module_count AS "live_gift_card_module_count"
            FROM ANALYTICS_CORE_ARR.MONTHLY_CUSTOMER_LIVE_MODULES m
            INNER JOIN ANALYTICS_CORE.CUSTOMER r ON m.customer_id = r.customer_id
            INNER JOIN (SELECT DISTINCT YR_MO, FIRSTDAYOFMONTH
                        FROM ANALYTICS_CORE.DATE_DIM
                        WHERE DATE_ID between {} and {}) as d
            ON d.YR_MO = m.YR_MO
            WHERE r.FIRST_ORDER_DATE IS NOT NULL
            AND r.ACCOUNT_TOAST_ID IS NOT NULL;
'''
monthly_modules = DataWarehouseConnection().query(sql_module.format(ts_start_date, end_date))
monthly_modules['live_saas_mrr'] = monthly_modules['live_saas_mrr'].astype(float)
monthly_modules.rename(columns={'restaurant_id': config.GROUP}, inplace=True)
monthly_modules.to_parquet(os.path.join(output_dir, 'monthly_modules.parquet'))

In [8]:
## daily revenue data
rid_ls = accts[config.GROUP].tolist()
rid_group_dict = split_ids_by_range(rid_ls)
for k, v in rid_group_dict.items():
    LOGGER.info('...loading rid chunk %s', k)
    daily_ = DataWarehouseConnection().get_daily_rev(
        start_yyyymmdd=ts_start_date,
        end_yyyymmdd=end_date,
        rest_ids=list(map(str, v)),
    )
    daily_.rename(columns={'restaurant_id': config.GROUP}, inplace=True)
    if len(daily_) > 0:
        LOGGER.info('...saving rid chunk %s', k)
        daily_.to_parquet(os.path.join(transaction_output_dir, f'rid-{k}.parquet'))

...loading rid chunk 200000000000000000
...saving rid chunk 200000000000000000
...loading rid chunk 30000000000000000
...saving rid chunk 30000000000000000
...loading rid chunk 110000000000000000
...saving rid chunk 110000000000000000
...loading rid chunk 20000000000000000
...saving rid chunk 20000000000000000
...loading rid chunk 130000000000000000
...saving rid chunk 130000000000000000
...loading rid chunk 50000000000000000
...saving rid chunk 50000000000000000
...loading rid chunk 60000000000000000
...saving rid chunk 60000000000000000
...loading rid chunk 140000000000000000
...saving rid chunk 140000000000000000
...loading rid chunk 80000000000000000
...saving rid chunk 80000000000000000
...loading rid chunk 120000000000000000
...saving rid chunk 120000000000000000
...loading rid chunk 170000000000000000
...saving rid chunk 170000000000000000
...loading rid chunk 40000000000000000
...saving rid chunk 40000000000000000
...loading rid chunk 70000000000000000
...saving rid chunk 70000

/home/ec2-user/SageMaker/toast-cap-new/toast_cap/utilities/dao.py:999: UserWarning: Empty result set for batch: ['260258000000000000', '260009000000000000']
  warnings.warn("Empty result set for batch: %r" % in_list_batch)


In [9]:
## The following two queries may not be relevant for your project

## get Capital application dates for model validation
sql_capital_validation = """
SELECT entity_identifier as "entity_identifier", 
    cast(toastorders_restaurant_id as int) as "rid",
    TO_DATE(funding_requested_date_time) as "funding_requested_date",
    term as "term",
    is_pre_90
FROM toast_capital.loan_application_record
WHERE funding_requested_date_time is not null 
    AND funding_requested_date_time>='2022-01-01'
    AND toastorders_restaurant_id is not null
    AND is_pre_90 = False
    ;
"""

df_capital_validation = DataWarehouseConnection().query(sql_capital_validation)
df_capital_validation['funding_requested_date'] = pd.to_datetime(df_capital_validation['funding_requested_date'])
df_capital_validation.to_parquet(os.path.join(output_dir, 'capital_validation_population.parquet'))
df_capital_validation.head(1)

,entity_identifier,rid,funding_requested_date,term,IS_PRE_90
0,817586a6-3894-44fe-9284-24754e0a8588,132617000000000000,2025-01-06,360,False


In [10]:
df_capital_validation.IS_PRE_90.value_counts(dropna=False)

False    58887
Name: IS_PRE_90, dtype: int64

In [11]:
## The following two queries may not be relevant for your project

## get Capital application dates for model validation
sql_loans = """
SELECT entity_identifier as "entity_identifier", 
    cast(toastorders_restaurant_id as int) as "rid",
    origination_amount,
    product_type,
    default_date,
    default_reasons,
    is_pre_90
FROM TOAST.TOAST_CAPITAL.PRODUCT_RECORD
WHERE toastorders_restaurant_id is not null
    ;
"""

df_loans = DataWarehouseConnection().query(sql_loans)
df_loans.to_parquet(os.path.join(output_dir, 'capital_loans_population.parquet'))
df_loans.head(1)

,entity_identifier,rid,ORIGINATION_AMOUNT,PRODUCT_TYPE,DEFAULT_DATE,DEFAULT_REASONS,IS_PRE_90
0,74dc07db-e9ba-4864-a891-0e0a94d9ab7e,111663000000000000,1164.67,LEASE,None,None,None


In [12]:
df_loans = df_loans[df_loans['IS_PRE_90'] != True]
df_loans['IS_PRE_90'].value_counts(dropna=False)

None     50211
False    43435
Name: IS_PRE_90, dtype: int64

In [13]:
## The following two queries may not be relevant for your project

## get Capital application dates for model validation
sql_loans = """
SELECT entity_identifier as "entity_identifier", 
    cast(toastorders_restaurant_id as int) as "rid",
    origination_amount,
    product_type,
    default_date,
    default_reasons,
    is_pre_90
FROM TOAST.TOAST_CAPITAL.PRODUCT_RECORD
WHERE toastorders_restaurant_id is not null
    ;
"""

df_loans = DataWarehouseConnection().query(sql_loans)
df_loans.to_parquet(os.path.join(output_dir, 'capital_loans_population.parquet'))
df_loans.head(1)

,entity_identifier,rid,ORIGINATION_AMOUNT,PRODUCT_TYPE,DEFAULT_DATE,DEFAULT_REASONS,IS_PRE_90
0,74dc07db-e9ba-4864-a891-0e0a94d9ab7e,111663000000000000,1164.67,LEASE,None,None,None


In [7]:
from toast_cap.utilities import config
from pyathena.pandas.cursor import PandasCursor
from scone.aws.athena import pyathena_connection
from scone.aws.session import session_manager
session_name = config.PYATHENA_PROD
role_arn = config.AWS_IAM_PROD
athena_output_dir = config.ATHENA_S3_PROD

session_manager.assume_role(role_arn, session_name)
conn = pyathena_connection(
    session_manager.session,
    s3_dir=athena_output_dir,
    region_name='us-east-1',
    cursor_class=PandasCursor
)

In [8]:
def orders_query(test, start: str, end: str) -> str:
    """
        This function selects data from toastordersbase_mixed.toastordersbasedf
        to calculate total order and amount sum for every resturant id
        Args
        ----
        test
            whether is it preprod
        start
            beginning of the date range
        end
            end of the date range
        Returns
        -------
            Sql query script for restaurant id and its net revenue ratio
    """
    if test:
        key = 'preproduction'
    else:
        key = 'production'
        
    sql = f"""SELECT 
    restaurant_guid, 
    yyyymmdd,
    amount_sum,
    CASE WHEN total_orders > 3000 THEN 3000 ELSE total_orders END as total_orders
    FROM (
        SELECT
            restaurant_guid, 
            yyyymmdd,
            SUM(amount) + SUM(tip) as amount_sum,
            SUM(captured_payment) as total_orders
        FROM (
            SELECT
                yyyymmdd,
                restaurant_guid, 
                ordrpymnt_amount * captured_payment as amount, 
                ordrpymnt_tipAmount * captured_payment as tip,
                captured_payment
            FROM (
                SELECT
                    yyyymmdd,
                    _restaurant_guid as restaurant_guid,
                    ordrpymnt_amount,
                    ordrpymnt_tipAmount,
                    (CASE WHEN ordrpymnt_paymentstatus = 'CAPTURED' or ordrpymnt_paymentstatus = 'AUTHORIZED' or ordrpymnt_paymentstatus = 'CAPTURE_IN_PROGRESS'  THEN 1 ELSE 0 END) as captured_payment
                FROM {key}_toastordersbase_mixed.toastordersbasedf
                WHERE recordtype = 'ORDER_PAYMENT' 
                AND ordrpymnt_paymentstatus IN ('CAPTURED', 'AUTHORIZED', 'CAPTURE_IN_PROGRESS')
                AND CASE
           WHEN ordrpymnt_paymentstatus = 'AUTHORIZED'
               AND ordrpymnt_paymenttype = 'CREDIT'
               THEN date_diff('day', current_date, ordrpymnt_paiddate_timestamp)
           ELSE 0
           END <= 9
                AND ordrpymnt_deleted = FALSE
                AND ordrpymnt_amount > 0
                AND yyyymmdd >= {start} 
                AND yyyymmdd <= {end}
            )
        )
        GROUP BY restaurant_guid, yyyymmdd
        )"""
    
    return sql


def load_orders_data(
    start_yyyymmdd: str, end_yyyymmdd: str, conn, accts_rid_guid_mapping, test: bool = True
) -> pd.DataFrame:
    """
    This function loads orders data using athena
    Parameters
    ----------
    start_yyyymmdd
        start date string, e.g. '20220101'
    end_yyyymmdd
        end date string, e.g. '20220901'
    test : bool
        if True, retrieve data from preprod
    conn : pyathena connection
    Returns
    -------
    Orders_data df: Amount sum and numbe of orders for each restaurant
    """

    # Get orders data
    orders_sql = orders_query(
        test=test, start=start_yyyymmdd, end=end_yyyymmdd
    )
    # Execute the query
    orders_df = conn.cursor().execute(orders_sql).as_pandas()
    # join with account table for rid
    orders_df = pd.merge(orders_df,accts_rid_guid_mapping, on='restaurant_guid', how='inner')
    orders_df = orders_df.drop(columns=['restaurant_guid'])
    # convert to float
    orders_df[config.GROUP] = orders_df[config.GROUP].astype('int64')
    orders_df['yyyymmdd'] = pd.to_datetime(orders_df.yyyymmdd,format="%Y%m%d")
    orders_df["amount_sum"] = orders_df.amount_sum.astype('float64')
    orders_df["total_orders"] = orders_df.total_orders.astype('float64')
    return orders_df

In [9]:
orders_sql = orders_query(
        test=False, start=volume_start_dt, end=run_date_str
    )
df_orders = conn.cursor().execute(orders_sql).as_pandas()

In [10]:
df_orders = pd.merge(df_orders, df_rid_guid_bridge, on='restaurant_guid', how='inner')

# df_orders = df_orders.drop(columns=['restaurant_guid'])
# convert to float
df_orders[config.GROUP] = df_orders[config.GROUP].astype('int64')
df_orders['yyyymmdd'] = pd.to_datetime(df_orders.yyyymmdd,format="%Y%m%d")
df_orders["amount_sum"] = df_orders.amount_sum.astype('float64')
df_orders["total_orders"] = df_orders.total_orders.astype('float64')

In [11]:
df_orders = df_orders.set_index('yyyymmdd')\
.groupby('restaurant_guid')\
.apply(lambda x: x.drop('restaurant_guid',axis=1)\
    .resample('D').asfreq().fillna(0))\
.reset_index()
df_orders = df_orders.drop(columns=['restaurant_guid'])
df_orders.head()

,yyyymmdd,amount_sum,total_orders,rid
0,2020-08-10,578.83,46.0,5.031200e+16
1,2020-08-11,450.05,35.0,5.031200e+16
2,2020-08-12,321.24,27.0,5.031200e+16
3,2020-08-13,374.78,29.0,5.031200e+16
4,2020-08-14,717.63,65.0,5.031200e+16


In [12]:
df_orders.to_parquet(os.path.join(output_dir, 'orders.parquet'))

In [13]:
df_orders.head()

,yyyymmdd,amount_sum,total_orders,rid
0,2020-08-10,578.83,46.0,5.031200e+16
1,2020-08-11,450.05,35.0,5.031200e+16
2,2020-08-12,321.24,27.0,5.031200e+16
3,2020-08-13,374.78,29.0,5.031200e+16
4,2020-08-14,717.63,65.0,5.031200e+16


# Payroll Bounce

In [23]:
df_pb_20242025 = pd.read_excel('/home/ec2-user/SageMaker/toast-cap-new/notebooks/Toast Payroll - Bounced Payrolls.xlsx', sheet_name='20242025 Transaction List')
df_pb_2023 = pd.read_excel('/home/ec2-user/SageMaker/toast-cap-new/notebooks/Toast Payroll - Bounced Payrolls.xlsx', sheet_name='2023 Transaction List')
df_pb_2022 = pd.read_excel('/home/ec2-user/SageMaker/toast-cap-new/notebooks/Toast Payroll - Bounced Payrolls.xlsx', sheet_name='2022 Transaction List')
df_pb_2021 = pd.read_excel('/home/ec2-user/SageMaker/toast-cap-new/notebooks/Toast Payroll - Bounced Payrolls.xlsx', sheet_name='2021 Transaction List')
df_pb_2020 = pd.read_excel('/home/ec2-user/SageMaker/toast-cap-new/notebooks/Toast Payroll - Bounced Payrolls.xlsx', sheet_name='2020 Transaction List')
df_pb_2019 = pd.read_excel('/home/ec2-user/SageMaker/toast-cap-new/notebooks/Toast Payroll - Bounced Payrolls.xlsx', sheet_name='2019 Transaction List')
df_bridge = pd.read_excel('/home/ec2-user/SageMaker/toast-cap-new/notebooks/Toast Payroll - Bounced Payrolls.xlsx', sheet_name='Restaurant GUID to Payroll Comp')

/home/ec2-user/SageMaker/.persisted_conda/toast-cap/lib/python3.9/site-packages/openpyxl/worksheet/_reader.py:223: UserWarning: Cell V406 is marked as a date but the serial value 6619688.0 is outside the limits for dates. The cell will be treated as an error.
  warn(msg)
/home/ec2-user/SageMaker/.persisted_conda/toast-cap/lib/python3.9/site-packages/openpyxl/worksheet/_reader.py:223: UserWarning: Cell V1640 is marked as a date but the serial value 21959878.0 is outside the limits for dates. The cell will be treated as an error.
  warn(msg)
/home/ec2-user/SageMaker/.persisted_conda/toast-cap/lib/python3.9/site-packages/openpyxl/worksheet/_reader.py:223: UserWarning: Cell V4538 is marked as a date but the serial value 6985049.0 is outside the limits for dates. The cell will be treated as an error.
  warn(msg)
/home/ec2-user/SageMaker/.persisted_conda/toast-cap/lib/python3.9/site-packages/openpyxl/worksheet/_reader.py:223: UserWarning: Cell V6771 is marked as a date but the serial value 6

In [229]:
df_pb = pd.concat([df_pb_20242025,df_pb_2023,df_pb_2022,df_pb_2021,df_pb_2020])
df_pb = df_pb[df_pb['Date'].notna()]

In [230]:
df_pb.columns = df_pb.columns.str.lower().str.replace(" ", "_")
df_bridge.columns = df_bridge.columns.str.lower().str.replace(" ", "_")

In [231]:
df_pb = pd.merge(df_pb,df_bridge[['company_code','restaurant_guid']],how='inner',on=['company_code'])

In [232]:
df_pb.rename(columns={'date':'payroll_bounce_date','effective_date':'payroll_calendar_date'},inplace=True)

In [233]:
list_drop = ['number_od','date_escrow_swept','fee_income_date','bounced_pay_reason_(see_bottom_for_key)','case:',\
             'notes/follow_up:','feedback','amount_in_tax_escrow','age_-_days','collection_status','last_updated','deduplication','net_pay_total','outstanding_fee']
df_pb = df_pb.drop(columns=list_drop)

In [234]:
df_pb['company_code'] = df_pb['company_code'].astype(str)
df_pb['client'] = df_pb['client'].astype(str) 
df_pb['payroll_bounce_date'] = pd.to_datetime(df_pb['payroll_bounce_date'], format='%Y-%m-%d', errors='coerce')
df_pb['payroll_calendar_date'] = pd.to_datetime(df_pb['payroll_calendar_date'], format='%Y-%m-%d', errors='coerce') 
df_pb['payment_date'] = pd.to_datetime(df_pb['payment_date'], format='%Y-%m-%d', errors='coerce')

# invalid_rows = df_pb[df_pb['net_pay_total'].apply(lambda x: isinstance(x, datetime.datetime))]
# df_pb = df_pb[~df_pb['net_pay_total'].apply(lambda x: isinstance(x, datetime.datetime))]
# df_pb['net_pay_total'] = df_pb['net_pay_total'].astype(float)

# invalid_rows = df_pb[df_pb['outstanding_fee'].str.contains(r'\.\.', regex=True, na=False)]
# df_pb = df_pb[~df_pb['outstanding_fee'].str.contains(r'\.\.', regex=True, na=False)]
# df_pb['case:'] = pd.to_numeric(df_pb['case:'], errors='coerce')

df_pb['reason_for_bounce'] = df_pb['reason_for_bounce:']
df_pb=df_pb.drop(columns = ['reason_for_bounce:'])

df_pb['restaurant_guid'] = df_pb['restaurant_guid'].apply(lambda x: str(x) if isinstance(x, int) else x)

In [268]:
df_pb.to_parquet(os.path.join(output_dir, 'payroll_bounce.parquet'))

In [267]:
df_pb.columns

Index(['payroll_bounce_date', 'company_code', 'client',
       'payroll_calendar_date', 'taxes_total', 'pay_period/group',
       'psr_line_item:', 'bounced_amount', 'fee', 'debit_total', 'wire_total',
       'pending_reversals', 'successful_reversals', 'distribution_adhoc'd',
       'reverse_write-off:_client_initiated', 'reverse_write-off:_retry_debit',
       'reverse_write-off:_garnishment', 'write_off', 'escrow_swept',
       'payment_date', 'current_outstanding', 'reverse_write-off_date',
       'inactive_in_mt', 'tpc_client_y/n', 'ewa_access_restored?',
       'date_ewa_notice_scheduled', 'exlclude_from_manual_retries',
       'company_code.1', 'write-off_candidate', 'active_in_mt',
       'repeat_offender', 'date_code', 'restrictions_imposed?',
       'restaurant_guid', 'reason_for_bounce'],
      dtype='object')

In [259]:
df_pd = df_pb.rename(columns=lambda x: x.strip())  # Removes any leading/trailing spaces or special characters

In [266]:
df_pb['reason_for_bounce'].value_counts()

Insufficient Funds              24343
Unauthorized Corporate Debit     2743
Account Frozen                    741
Account Closed                    667
Split Payment                     658
Payment Stopped                   569
No Account/Cannot Locate          438
Invalid ACH Routing Number        261
Invalid Account Number            241
Non Transaction Acct              178
No Acct/Cannot Locate             172
Uncollected Funds                 165
T/R Check Digit Error              82
NO ACCT/CANNOT LOCATE              75
NO ACCT/CANNOT LOCATE\n            53
NON TRANSACTION ACCT               39
Cust Advises Incorrect             37
NON TRANSACTION ACCT\n             28
UNCOLLECTED FUNDS                  25
No Account/ Cannot Locate          24
Non Transaction Account            14
Duplicate Entry                    11
CORP ENTRY TO CONSUMER              9
tdog                                9
Split payment                       8
split payment                       7
CORP ENTRY T

# MidDesk Data

In [240]:
## The following two queries may not be relevant for your project

## get Capital application dates for model validation
sql_middesk = """
SELECT toastorders_restaurant_guid, 
funding_requested_date_time,
status,
parse_json(application_record_json),
parse_json(application_record_json)['evaluationDetails']['bankruptcies'] as bankruptcies,
parse_json(application_record_json)['evaluationDetails']['lienExternalLink'] as lienExternalLink,
parse_json(application_record_json)['evaluationDetails']['liens'] as liens,
parse_json(application_record_json)['evaluationDetails']['liens']['amountUSD'] as lien_amount,
parse_json(application_record_json)['evaluationDetails']['liens']['fileDate'] as lien_date,
application_record_json['evaluationDetails']['businessFormationDate'] as businessFormationDate,
application_record_json['evaluationDetails']['businessEntityType'] as businessEntityType
FROM TOAST.TOAST_CAPITAL.LOAN_APPLICATION_RECORD
order by yyyymmdd desc
    ;
"""

df_middesk = DataWarehouseConnection().query(sql_middesk)
df_middesk.to_parquet(os.path.join(output_dir, 'middesk.parquet'))
df_middesk.columns = df_middesk.columns.str.lower().str.replace(" ", "_")
df_middesk.head(1)

,toastorders_restaurant_guid,funding_requested_date_time,status,parse_json(application_record_json),bankruptcies,lienexternallink,liens,lien_amount,lien_date,businessformationdate,businessentitytype
0,08d1e777-33ae-481b-a5f0-cc13771a6e9a,2025-03-11 18:52:41.899000-04:00,ACTIVE_LOAN,"{\n ""applicationInfo"": {\n ""applicantEmail...",[],"""https://app.middesk.com/businesses/dfd75b04-4...",[],None,None,20240814,"""LLC"""


In [246]:
df_middesk.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 76387 entries, 0 to 76386
Data columns (total 11 columns):
 #   Column                               Non-Null Count  Dtype 
---  ------                               --------------  ----- 
 0   toastorders_restaurant_guid          70255 non-null  object
 1   funding_requested_date_time          74510 non-null  object
 2   status                               76387 non-null  object
 3   parse_json(application_record_json)  70255 non-null  object
 4   bankruptcies                         41635 non-null  object
 5   lienexternallink                     45030 non-null  object
 6   liens                                51068 non-null  object
 7   lien_amount                          0 non-null      object
 8   lien_date                            0 non-null      object
 9   businessformationdate                2257 non-null   object
 10  businessentitytype                   2257 non-null   object
dtypes: object(11)
memory usage: 6.4+ MB


In [274]:
df_middesk.head().T

,0,1,2,3,4
toastorders_restaurant_guid,08d1e777-33ae-481b-a5f0-cc13771a6e9a,f2d82b63-2f38-407f-b856-0274565602a4,f69d84b0-5980-47b3-81e8-bdb32e61c195,3f086941-625e-401a-94c8-bef6272d4455,93bdc465-e48d-464d-96c3-d431cfd904b6
funding_requested_date_time,2025-03-11 18:52:41.899000-04:00,2025-03-12 19:35:04.615000-04:00,2025-03-11 20:16:24.166000-04:00,2025-03-11 19:21:30.532000-04:00,2025-03-12 17:54:20.784000-04:00
status,ACTIVE_LOAN,DOES_NOT_QUALIFY,CONTINGENCY,ACTIVE_LOAN,DOES_NOT_QUALIFY_MANUAL_REVIEW
parse_json(application_record_json),"{\n ""applicationInfo"": {\n ""applicantEmail...","{\n ""applicationInfo"": {\n ""applicantEmail...","{\n ""applicationInfo"": {\n ""applicantEmail...","{\n ""applicationInfo"": {\n ""applicantEmail...","{\n ""applicationInfo"": {\n ""applicantEmail..."
bankruptcies,[],[],[],[],[]
lienexternallink,"""https://app.middesk.com/businesses/dfd75b04-4...","""https://app.middesk.com/businesses/6f55a7b1-6...","""https://app.middesk.com/businesses/0b47b9f5-3...","""https://app.middesk.com/businesses/d7d5cefd-b...","""https://app.middesk.com/businesses/f07f0769-6..."
liens,[],[],"[\n {\n ""amountUSD"": 0,\n ""collateral"":...","[\n {\n ""amountUSD"": 0,\n ""collateral"":...",[]
lien_amount,None,None,None,None,None
lien_date,None,None,None,None,None
businessformationdate,20240814,null,20140103,null,null


# Payroll Data

In [295]:

## get Capital application dates for model validation
sql_xtrachef = """
SELECT 
    distinct collocation_id,toast_restaurant_guid 
FROM 
    TOAST.CS_ONBOARDING.xtrachef_onboarding_events
    ;
"""

df_map = DataWarehouseConnection().query(sql_xtrachef)
# df_xtrachef.to_parquet(os.path.join(output_dir, 'capital_loans_population.parquet'))
df_map.columns = df_map.columns.str.lower().str.replace(" ", "_")
df_map.head(1)

,collocation_id,toast_restaurant_guid
0,38357,6de63fa1-b244-4f72-975d-e219d4ef91aa


In [296]:
sql_payroll_current = '''
SELECT 
    distinct CUSTOMER_UUID
FROM 
    TOAST.SOURCE_EC.PAYROLL_EARNING_CURRENT
 
'''
df_pay = DataWarehouseConnection().query(sql_payroll_current)
# df_xtrachef.to_parquet(os.path.join(output_dir, 'capital_loans_population.parquet'))
df_pay.columns = df_pay.columns.str.lower().str.replace(" ", "_")
df_pay.head(1)

,customer_uuid
0,4d5a92ed-9064-11ea-b09c-0aacafde20f8


In [297]:
df_pay.shape

(31529, 1)

In [298]:
sql_cust_current = '''
SELECT 
    *
FROM
    TOAST.SOURCE_EC.CUSTOMER_CURRENT
'''
df_cust = DataWarehouseConnection().query(sql_cust_current)
# df_xtrachef.to_parquet(os.path.join(output_dir, 'capital_loans_population.parquet'))
df_cust.columns = df_cust.columns.str.lower().str.replace(" ", "_")
df_cust.head(1)

,yyyymmdd,collocation_id,entity_identifier,version,uuid,name,company_code,status
0,2009,5574100407046902250,"[77, 91, 46, 201, 144, 100, 17, 234, 176, 156,...",1708934948,4d5b2ec9-9064-11ea-b09c-0aacafde20f8,NA Corporation of Illinois,NAC,INACTIVE


In [300]:
df_cust.shape

(49777, 8)

In [301]:
df_cust_pay = pd.merge(df_cust,df_pay,left_on = 'uuid',right_on = 'customer_uuid',how='inner')

In [302]:
df_cust_pay.shape

(31529, 9)

In [309]:
df_pb_match_cp = df_cust_pay.drop_duplicates(subset=['uuid'],keep='last')
df_pb_match_cp['company_code'].shape

(31529,)

In [308]:
df_cust_pay.head(1)

,yyyymmdd,collocation_id,entity_identifier,version,uuid,name,company_code,status,customer_uuid
0,2009,5574100407046902250,"[77, 91, 46, 201, 144, 100, 17, 234, 176, 156,...",1708934948,4d5b2ec9-9064-11ea-b09c-0aacafde20f8,NA Corporation of Illinois,NAC,INACTIVE,4d5b2ec9-9064-11ea-b09c-0aacafde20f8


In [303]:
df_cust_pay['status'].value_counts(dropna=False)

ACTIVE                      24158
NEW_CLIENT_MID_QUARTER       4538
INACTIVE                     1918
NEW_CLIENT_CLEAN_QUARTER      890
DEMO                           25
Name: status, dtype: int64

In [286]:
accts.shape

(183518, 18)

In [304]:
df_pb_match = pd.merge(df_cust_pay,df_pb,on='company_code',how='inner')
df_pb_match.shape

(27748, 43)

In [306]:
df_pb_match_cp = df_pb_match.drop_duplicates(subset=['company_code'],keep='last')
df_pb_match_cp['company_code'].shape

(3502,)

In [312]:
df_cust_w_pay =  pd.merge(df_pb_match,accts,left_on='restaurant_guid',right_on='restaurant_guid',how='inner')
df_cust_w_pay.shape

(27430, 60)

In [314]:
df_pb_match_cp = df_cust_w_pay.drop_duplicates(subset=['restaurant_guid'],keep='last')
df_pb_match_cp['restaurant_guid'].shape

(5347,)

In [307]:
df_pb_match_cp = df_pb.drop_duplicates(subset=['company_code'],keep='last')
df_pb_match_cp['company_code'].shape

(3939,)

In [ ]:
pd.merge(current_rid.astype(str),df_final['rid'],left_on='rid',right_on='rid',how='inner').rid.nunique()